In [1]:
import json
import numpy as np
import pandas as pd
from io import StringIO
import textwrap
from model_inference.gpt import *
from utils.table_utils import *

# Table parsing test

In [2]:
path = '../data/livesum/test.json'

In [3]:
df = pd.read_json(path)

In [4]:
print(df.head())

                                                text  \
0  And we're off for the first half. Player27(Awa...   
1  The game is underway with the start of the fir...   
2  The game is underway with the start of the fir...   
3  And we're off for the first half. Player26(Awa...   
4  The game is underway with the start of the fir...   

                                               table        id  
0  Team,Goals,Shots,Fouls,Yellow Cards,Red Cards,...  25513332  
1  Team,Goals,Shots,Fouls,Yellow Cards,Red Cards,...  25513360  
2  Team,Goals,Shots,Fouls,Yellow Cards,Red Cards,...  25600389  
3  Team,Goals,Shots,Fouls,Yellow Cards,Red Cards,...  25617902  
4  Team,Goals,Shots,Fouls,Yellow Cards,Red Cards,...  25892175  


In [5]:
print(df.columns)

Index(['text', 'table', 'id'], dtype='object')


In [6]:
idx = 4

In [7]:
print(df['table'][idx])

Team,Goals,Shots,Fouls,Yellow Cards,Red Cards,Corner Kicks,Free Kicks,Offsides<NEWLINE>Away Team,3,16,13,2,0,3,18,0<NEWLINE>Home Team,0,28,18,2,0,7,12,2


In [8]:
table_string = df['table'][idx]
table_string = table_string.replace('<NEWLINE>', '\n')

In [9]:
table_string_io = StringIO(table_string)

In [10]:
df_table = pd.read_csv(table_string_io)

In [11]:
print(df_table.to_string(index=False))

     Team  Goals  Shots  Fouls  Yellow Cards  Red Cards  Corner Kicks  Free Kicks  Offsides
Away Team      3     16     13             2          0             3          18         0
Home Team      0     28     18             2          0             7          12         2


# Prompting test

In [12]:
df = pd.read_json('../data/livesum/test.json')
%clear
print(textwrap.fill(df['text'][idx], width=100))

The game is underway with the start of the first half. Jozy Player27(Away Team) commits a foul on
Player4(Home Team), who earns a free kick on the right wing, The foul results in a free kick for
Player4(Home Team) on the right wing. Player4(Home Team) misses the goal with a right-footed shot
from outside the box after a set piece. Player2(Home Team) misses the goal with a right-footed shot
from the right side of the box, assisted by Player7(Home Team). Player8(Home Team) commits a foul.
Player20(Away Team) earns a free kick on the right side of the field. Player21(Away Team)'s header
from close range was just slightly too high after being assisted by Player25(Away Team) with a cross
from a set piece. Player7(Home Team) commits a foul, resulting in Player21(Away Team) winning a free
kick in their own defensive half. Player4(Home Team) commits a foul on Player28(Away Team),
resulting in a free kick for the defensive team. Player8(Home Team) attempts a through ball, but
Player7(Home Team)

In [13]:
text = df['text'][idx]
atomic_out = ask_chatgpt(text=text,prompt_path="prompts/livesum_atomic.txt")
print(atomic_out)

Jozy Player27 (Away Team) commits a foul on Player4 (Home Team).  
Player4 (Home Team) earns a free kick on the right wing.  
Player4 (Home Team) misses the goal with a right-footed shot from outside the box after a set piece.  
Player2 (Home Team) misses the goal with a right-footed shot from the right side of the box, assisted by Player7 (Home Team).  
Player8 (Home Team) commits a foul.  
Player20 (Away Team) earns a free kick on the right side of the field.  
Player21 (Away Team) has a header from close range that is slightly too high after being assisted by Player25 (Away Team) with a cross from a set piece.  
Player7 (Home Team) commits a foul, resulting in Player21 (Away Team) winning a free kick in their own defensive half.  
Player4 (Home Team) commits a foul on Player28 (Away Team), resulting in a free kick for the defensive team.  
Player8 (Home Team) attempts a through ball, but Player7 (Home Team) is offside.  
Player2 (Home Team) commits a foul.  
Player29 (Away Team) ear

In [14]:
with open('./model_outputs/gpt_livesum_test/atomic_each.txt', 'w') as f:
    f.write(atomic_out)

In [15]:
with open('./model_outputs/gpt_livesum_test/atomic_each.txt', 'r') as f:
    atomic_text = f.read()
header_out = ask_chatgpt(text=atomic_text,prompt_path="prompts/livesum_header.txt")
print(header_out)

{
  "row_headers": [
    "Player4 (Home Team)",
    "Player2 (Home Team)",
    "Player3 (Home Team)",
    "Player6 (Home Team)",
    "Player8 (Home Team)",
    "Player9 (Home Team)",
    "Player10 (Home Team)",
    "Player7 (Home Team)",
    "Player5 (Home Team)",
    "Player13 (Home Team)",
    "Player16 (Home Team)",
    "Player17 (Home Team)",
    "Player21 (Away Team)",
    "Player22 (Away Team)",
    "Player23 (Away Team)",
    "Player24 (Away Team)",
    "Player25 (Away Team)",
    "Player26 (Away Team)",
    "Player27 (Away Team)",
    "Player28 (Away Team)",
    "Player29 (Away Team)",
    "Player30 (Away Team)",
    "Player20 (Away Team)",
    "Player9 (Home Team)"
  ],
  "column_headers": [
    "Fouls Committed",
    "Free Kicks Earned",
    "Shots Attempted",
    "Goals Scored",
    "Yellow Cards",
    "Corner Kicks Earned",
    "Penalties Awarded",
    "Saves Made",
    "Offsides Called"
  ]
}


In [16]:
with open('./model_outputs/gpt_livesum_test/header_each.txt', 'w') as f:
    f.write(header_out)

In [17]:
with open('./model_outputs/gpt_livesum_test/header_each.txt', 'r') as f:
    header_text = f.read()
with open('./model_outputs/gpt_livesum_test/atomic_each.txt', 'r') as f:
    atomic_text = f.read()
input_text = header_text + '\n' + atomic_text
output_table = ask_chatgpt(text=input_text,prompt_path="prompts/livesum_table.txt")
print(output_table)

|  | Fouls Committed | Free Kicks Earned | Shots Attempted | Goals Scored | Yellow Cards | Corner Kicks Earned | Penalties Awarded | Saves Made | Offsides Called |
| Player4 (Home Team) | 5 | 3 | 5 | 0 | 0 | 3 | 0 | 0 | 0 |
| Player2 (Home Team) | 3 | 2 | 2 | 0 | 0 | 0 | 0 | 0 | 0 |
| Player3 (Home Team) | 2 | 1 | 4 | 0 | 1 | 2 | 0 | 0 | 0 |
| Player6 (Home Team) | 2 | 1 | 1 | 0 | 0 | 1 | 0 | 0 | 0 |
| Player8 (Home Team) | 3 | 1 | 3 | 0 | 1 | 1 | 1 | 1 | 1 |
| Player9 (Home Team) | 3 | 2 | 4 | 0 | 0 | 2 | 0 | 1 | 0 |
| Player10 (Home Team) | 2 | 1 | 2 | 0 | 0 | 1 | 0 | 1 | 0 |
| Player7 (Home Team) | 3 | 1 | 3 | 0 | 0 | 1 | 0 | 0 | 1 |
| Player5 (Home Team) | 2 | 1 | 2 | 0 | 0 | 0 | 0 | 0 | 0 |
| Player13 (Home Team) | 1 | 1 | 3 | 0 | 0 | 2 | 0 | 1 | 0 |
| Player16 (Home Team) | 1 | 0 | 1 | 0 | 1 | 0 | 0 | 0 | 0 |
| Player17 (Home Team) | 1 | 0 | 1 | 0 | 1 | 0 | 0 | 0 | 0 |
| Player21 (Away Team) | 1 | 1 | 2 | 0 | 0 | 0 | 0 | 0 | 0 |
| Player22 (Away Team) | 3 | 2 | 1 | 0 | 0 | 0 | 0 

In [18]:
convert_to_df(output_table)

,,Fouls Committed,Free Kicks Earned,Shots Attempted,Goals Scored,Yellow Cards,Corner Kicks Earned,Penalties Awarded,Saves Made,Offsides Called
0,Player4 (Home Team),5,3,5,0,0,3,0,0,0
1,Player2 (Home Team),3,2,2,0,0,0,0,0,0
2,Player3 (Home Team),2,1,4,0,1,2,0,0,0
3,Player6 (Home Team),2,1,1,0,0,1,0,0,0
4,Player8 (Home Team),3,1,3,0,1,1,1,1,1
5,Player9 (Home Team),3,2,4,0,0,2,0,1,0
6,Player10 (Home Team),2,1,2,0,0,1,0,1,0
7,Player7 (Home Team),3,1,3,0,0,1,0,0,1
8,Player5 (Home Team),2,1,2,0,0,0,0,0,0
9,Player13 (Home Team),1,1,3,0,0,2,0,1,0


In [19]:
print(df_table.to_string(index=False))

     Team  Goals  Shots  Fouls  Yellow Cards  Red Cards  Corner Kicks  Free Kicks  Offsides
Away Team      3     16     13             2          0             3          18         0
Home Team      0     28     18             2          0             7          12         2
